In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/07/25 16:37:00] INFO     Found credentials from IAM Role:                                   ]8;id=872670;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=761559;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'gen-xii-parser'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# functions.py
COPY functions.py ${LAMBDA_TASK_ROOT}

# functions_counters.py
COPY functions_counters.py ${LAMBDA_TASK_ROOT}

# functions_counters_pricing.py
COPY functions_counters_pricing.py ${LAMBDA_TASK_ROOT}

# preprocessing.py
COPY preprocessing.py ${LAMBDA_TASK_ROOT}

# cls_parser.pkl
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Load files from Flask app

In [4]:
list_str_filenames = [
    'api.py',
    'cls_parser.pkl',
    'functions.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
    'preprocessing.py',
    'requirements.txt',
]
for str_filename in list_str_filenames:
    str_source = f'../01_flask_app/app/{str_filename}'
    str_destination = f'./{str_filename}'
    shutil.copyfile(str_source, str_destination)

### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pickle
import pandas as pd 
pd.options.mode.chained_assignment = None # suppress warning

# lambda handler
def lambda_handler(event, context):
    # import parser
    print('Loading parser...')
    print('')
    cls_parse_payload = pickle.load(open('cls_parser.pkl', 'rb'))
    # get payload
    print('Getting request...')
    print('')
    try:
        dict_json_request = event['request']
    except KeyError:
        dict_json_request = event
    # parse payload
    print('Parsing payload...')
    cls_parse_payload.get_data(dict_json_request)
    cls_parse_payload.engineer_pmt_hx()
    cls_parse_payload.shared_preprocessing()
    cls_parse_payload.generate_predictions()
    cls_parse_payload.apply_policies()
    cls_parse_payload.apply_chime()
    cls_parse_payload.adverse_action()
    cls_parse_payload.counter_offers()
    cls_parse_payload.generate_output()
    # extract output
    print('Extracting output...')
    print('')
    dict_output = cls_parse_payload.dict_output
    # return output_final
    return dict_output['output_final']

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=gen-xii-parser

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 844B done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.8
#2 DONE 0.2s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/11] FROM public.ecr.aws/lambda/python:3.8@sha256:19c611b6736ef5d1a1403a51b5140446c78158fb832771cac7be12ed513c361b
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 3.48MB 0.0s done
#5 DONE 0.0s

#6 [ 5/11] COPY api.py /var/task
#6 CACHED

#7 [ 6/11] COPY functions.py /var/task
#7 CACHED

#8 [ 7/11] COPY functions_counters.py /var/task
#8 CACHED

#9 [ 8/11] COPY functions_counters_pricing.py /var/task
#9 CACHED

#10 [ 9/11] COPY preprocessing.py /var/task
#10 CACHED

#11 [ 4/11] RUN  pip3 install -r requirements.txt --target "/var/task"
#11 CACHED

#12 [ 3/11] COPY requirements.txt  .
#12 CACHED

#13 [ 2/11] RUN pip install --upgrade pip
#13 CA

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen-xii-parser' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xii-parser]
a145b8c0a10c: Preparing
d110bea937d2: Preparing
2c1ae27c0d52: Preparing
784d812311e7: Preparing
5a8fb33ead1f: Preparing
57c7873e6916: Preparing
097d278c944a: Preparing
cf8a60b441a4: Preparing
02e21ed6b2aa: Preparing
2c65e6cda9e2: Preparing
69063223dcc9: Preparing
3e0f7053d2d2: Preparing
e1b8ef616f15: Preparing
57c7873e6916: Waiting
884ba2d905e7: Preparing
7f2a4f045bc4: Preparing
814345d22610: Preparing
097d278c944a: Waiting
cf8a60b441a4: Waiting
02e21ed6b2aa: Waiting
2c65e6cda9e2: Waiting
69063223dcc9: Waiting
814345d22610: Waiting
884ba2d905e7: Waiting
e1b8ef616f15: Waiting
3e0f7053d2d2: Waiting
7f2a4f045bc4: Waiting
784d812311e7: Layer already exists
5a8fb33ead1f: Layer already exists
a145b8c0a10c: Layer already exists
2c1ae27c0d52: Layer already exists
d110bea937d2: Layer already exists
57c7873e6916: Layer already exists
cf8a60b441a4: Layer already exists
2c65e6cda9e2: Layer already exists
09

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

[03/07/25 16:37:03] INFO     Found credentials from IAM Role:                                   ]8;id=816559;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=625800;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 07 Mar 2025 16:37:04 GMT',
                                      'x-amzn-requestid': 'e9be6abb-3a91-43a0-b64f-5fdbcf86d46e'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'e9be6abb-3a91-43a0-b64f-5fdbcf86d46e',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=180, # 3 minutes
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '947063bf8c102e66bd9eb8eee51ad4e638120ef60e82807de31ff1217a9ea970',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:gen-xii-parser',
 'FunctionName': 'gen-xii-parser',
 'LastModified': '2025-03-07T16:37:04.424+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/gen-xii-parser'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1176',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 07 Mar 2025 16:37:05 GMT',
                                      'x-amzn-requestid': '3ab056b9-7bd3-4359-bce8-b1491b116467'},
                      'HTTPStatusCode': 201,
                      'RequestId': '3ab056b9-7bd3-4359-bce8-b1491b116467',
 

### Clean-up

In [11]:
list_str_filenames = [
    'functions.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
    'requirements.txt',
    'preprocessing.py',
    'cls_parser.pkl',
    'Dockerfile',
    'api.py',
    'lambda_function.py',
]
for str_file in list_str_filenames:
    try:
        os.remove(str_file)
    except:
        pass